# 09 · Evaluate everything (CPU)

**What this does:** scores every saved run (baselines, outside geocoders, our models) with one harness, and writes:

| Output | Where | What for |
|---|---|---|
| `results.parquet` | data folder, `runs/` | every metric for every model, test set and slice |
| `leaderboard.md` | repo, `eval/` | the headline table, ready for the README |
| `explorer.json` | repo, `docs/data/` | feeds the error explorer web page (`docs/index.html`) |

**Reading the metrics**

| Metric | Meaning |
|---|---|
| coverage | share of real addresses that got any answer |
| acc@100, @500, @1000 | share answered within that many metres; no answer counts as a miss |
| p50, p90 | median and 90th-percentile error over answered addresses |
| misroute | answer fell in a different pincode area from the true one (a parcel would go to the wrong hub) |
| wrong_side | among answers within 150 m: on the wrong side of the building's road |
| access_p50 | median distance from the answer to the building's road access point (what a rider needs) |
| abstain_p / abstain_r | on gibberish rows: precision and recall of saying "no answer" |

**oracle_reference** is not a model. It is the distance from the building to the best place *named in the text*, so it shows how much the text pins down. Lookup methods cannot beat it. A learned model can, by combining clues.

**Runtime:** a minute or two, CPU. Re-run it whenever a new model has saved predictions.

In [ ]:
# Setup (the same cell opens every notebook): fetch the project code, install what is missing, find the data folder.
import os, sys, subprocess
REPO = 'SiddharthaRaja/Navigate'
PIP  = ['h3', 'rapidfuzz', 'scikit-learn', 'tabulate']
on_cloud = os.path.exists('/content') or os.path.exists('/kaggle/working')
if on_cloud:
    CODE = '/content/Navigate' if os.path.exists('/content') else '/kaggle/working/Navigate'
    if not os.path.exists(f'{CODE}/bench'):
        token = os.environ.get('GH_TOKEN', '')
        try:                                   # only needed while the GitHub repo is private
            from google.colab import userdata; token = token or userdata.get('GH_TOKEN')
        except Exception: pass
        try:
            from kaggle_secrets import UserSecretsClient; token = token or UserSecretsClient().get_secret('GH_TOKEN')
        except Exception: pass
        url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
        r = subprocess.run(['git', 'clone', '--depth', '1', url, CODE], env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
        assert r.returncode == 0, 'Could not fetch the code. If the repo is private, add a GitHub token as the secret GH_TOKEN.'
    else:
        subprocess.run(['git', '-C', CODE, 'pull', '--ff-only'])
    if PIP: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PIP], check=True)
else:
    CODE = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
sys.path.insert(0, CODE)
import numpy as np, pandas as pd
from bench import env
P = env.setup('hyderabad')
print('running on', P['where'], '| data folder:', P['root'])

In [ ]:
from bench import explorer as ex, io, metrics as mt, pincode as pc
B, RUNS = P['bench'], P['runs']
test    = io.read(B, 'test')
targets = pd.read_parquet(f'{B}/targets.parquet')
pin_lookup, _ = pc.make_lookup(P['gaz'])

runs, results = [], []
for r in io.list_runs(RUNS):
    if r['pred'] is not None:                     # our models and open geocoders: score now
        truth = test[test.id.isin(r['pred'].id)] if r['kind'] == 'open' else test
        r['scores'] = mt.score_rows(truth, r['pred'], pin_lookup, targets)
    truth = test[test.id.isin(r['scores'].id)]
    results.append(mt.summarise(truth, r['scores'], r['model']))
    runs.append(r); print(f"{r['model']:22s} {r['kind']:6s} {len(truth):,} rows")

oracle = mt.scores_from_errors(test, test.oracle_err_m)
results.append(mt.summarise(test, oracle, 'oracle_reference').drop(columns=['abstain_p', 'abstain_r'], errors='ignore'))
runs.append(dict(name='oracle_reference', kind='reference', pred=None, scores=oracle,
                 about='Distance to the best place named in the text. Not a model: lookup methods cannot beat it, a learned model can.'))
R = pd.concat(results, ignore_index=True)
R = R[~((R.model == 'oracle_reference') & (R.coverage == 0))]          # real addresses have no oracle: nothing to show
R.to_parquet(f'{RUNS}/results.parquet', index=False)

## 1. Leaderboard
Note: outside geocoders are scored on the API subset only (no invented door numbers), so compare them with our models on that subset, not on the full sets. The second table does exactly that.

In [ ]:
pd.set_option('display.width', 250, 'display.max_rows', 300)
board = mt.leaderboard(R)
print(board.to_string())

api_ids = set(test[test.api_subset].id)
sub = [mt.summarise(test[test.id.isin(api_ids)], r['scores'][r['scores'].id.isin(api_ids)], r['name'], slices=False) for r in runs]
board_api = mt.leaderboard(pd.concat(sub, ignore_index=True))
print('\nSame rows for everyone (API subset):'); print(board_api.to_string())

## 2. Where the best models differ: slices
Change `MODELS` to the ones you want side by side.

In [ ]:
MODELS = [m for m in ['gazetteer_rules', 'tfidf_knn', 'h3_classifier', 'addressbind', 'llm_sft', 'google', 'oracle_reference'] if m in set(R.model)]
for sl in ['style', 'relation', 'ref_in_gazetteer', 'noise_level', 'pin_state', 'cold_depth', 'paraphrased', 'source', 'noise_op']:
    x = R[(R['slice'] == sl) & R.model.isin(MODELS)]
    if len(x): print(f"\n--- median error (m) by {sl}"); print(x.pivot_table(index=['split', 'value'], columns='model', values='p50').round(0).to_string())

## 3. Is a difference real? Confidence intervals
With 3,000 rows, small gaps can be luck. This prints a 95% interval for each model's median error (bootstrap: resample the rows 500 times).

In [ ]:
for r in runs:
    for sp in ['test_warm', 'test_cold']:
        e = r['scores'].merge(test[['id', 'split', 'abstain']], on='id')
        e = e[(e.split == sp) & ~e.abstain].err_m
        if e.notna().sum() > 30: print(f"{r['name']:22s} {sp:10s} median {e.median():8.0f} m   95% interval {mt.bootstrap_ci(e)}")

## 4. Write the files for the repo
Two small files are written into the code folder: `eval/leaderboard.md` and `docs/data/explorer.json`. On Colab, download them (folder icon on the left) and put them in the same places in your local repo, then commit. The explorer shows predicted points only for our own models and open geocoders; paid geocoders appear as distances.

In [ ]:
os.makedirs(f'{CODE}/eval', exist_ok=True)
with open(f'{CODE}/eval/leaderboard.md', 'w') as f:
    f.write(f"# Leaderboard (Hyderabad, {pd.Timestamp.now():%Y-%m-%d})\n\nFull test sets:\n\n{board.reset_index().to_markdown(index=False)}\n\n"
            f"API subset (same rows for every system):\n\n{board_api.reset_index().to_markdown(index=False)}\n")
public = test[test.source.isin(['synthetic', 'osm_addr', 'overture_addr'])]          # never export private rows
NOTE = f"{len(public):,} test addresses from {targets.building_id.nunique():,} target buildings."
size = ex.export(public, runs, R, f'{CODE}/docs/data/explorer.json', note=NOTE)
print(f"wrote eval/leaderboard.md and docs/data/explorer.json ({size / 1e6:.1f} MB) under {CODE}")
print('To view locally:  python -m http.server --directory docs 8000   then open http://localhost:8000')

## Share the outputs
Packs only the files this notebook created into one zip in `My Drive > Navigate > data > share`. Download that one file to share or back up the results.

In [ ]:
env.pack_outputs(P, '09' + '_outputs', [f"{P['runs']}/results.parquet", f'{CODE}/eval/leaderboard.md', f'{CODE}/docs/data/explorer.json'])